In [6]:
import numpy as np
import scipy.stats as stats
import src.noise as noise
import src.release as dprelease

from src.analysis import OddsRatio

Here's a contingency table.

In [7]:
true_tbl = np.array([
    [65, 109],
    [243, 1348]
])

It's straightforward to have `scipy` calculate the 95% confidence interval of the odds ratio.

In [8]:
scipy_ci = stats.contingency.odds_ratio(true_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.363633, 4.629785)


The `dpvalue` library can do this too. It implicitly converts `true_tbl` into a table with elements of type `NoisyFloat` (the noise is just zero).

In [9]:
dpvalue_lo, dpvalue_hi = OddsRatio(true_tbl).sample().confidence_interval(0.05)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.352972, 4.626609)


More or less consistent, I guess.

Let's add some noise and pretend the contingency table is a differentially private release.

In [10]:
noise_factory = noise.gaussian(loc=0.0, scale=3.0)
noisy_tbl = dprelease.noisy_float_array(true_tbl, noise_factory)
noisy_tbl

array([[NoisyFloat(obs=61.792779639906236, expr=id-1, thetas=frozenset({id-1}), eqns=(id-1 + id-0 - 61.7927796399062,)),
        NoisyFloat(obs=110.01878604717363, expr=id-3, thetas=frozenset({id-3}), eqns=(id-3 + id-2 - 110.018786047174,))],
       [NoisyFloat(obs=248.1057255030806, expr=id-5, thetas=frozenset({id-5}), eqns=(id-5 + id-4 - 248.105725503081,)),
        NoisyFloat(obs=1346.3345691531467, expr=id-7, thetas=frozenset({id-7}), eqns=(id-7 + id-6 - 1346.33456915315,))]],
      dtype=object)

What would `scipy` do with this? Since `scipy` isn't DP-noise-aware, it can only take the observed values as truth and account for sampling uncertainty as it did before.

But oops, we can't use `scipy` on this directly, even though `NoisyFloat` is castable to `float`. The elements are counts, and have to be type `int`. Let's extract the noisy observations.

In [ ]:
noisy_int_tbl = [
    [int(noisy_tbl[0, 0].obs), int(noisy_tbl[0, 1].obs)],
    [int(noisy_tbl[1, 0].obs), int(noisy_tbl[1, 1].obs)]]

noisy_int_tbl

[[61, 110], [248, 1346]]

And here's the `scipy` confidence interval again.

In [ ]:
scipy_ci = stats.contingency.odds_ratio(noisy_int_tbl, kind="sample").confidence_interval(confidence_level=0.95)
print(f"95% CI: ({scipy_ci.low:.6f}, {scipy_ci.high:.6f})")

95% CI: (2.140236, 4.232523)


But the `dpvalue` library uses `noisy_tbl` directly. It accounts for both uncertainty from DP noise _and_ uncertainty from sampling.

In [15]:
dpvalue_lo, dpvalue_hi = OddsRatio(noisy_tbl).sample().confidence_interval(0.05)
print(f"95% CI: ({dpvalue_lo:.6f}, {dpvalue_hi:.6f})")

95% CI: (2.119658, 4.320783)


And because of this, it knows the 95% odds ratio confidence interval for the DP release is actually a little bit wider.